# Study v2: protocol and source audit

This notebook checks the frozen design, the strict GPT-5.6 Sol source audit, and the separate packet
reserved for a future independent human audit. The model audit never writes into the human ledgers
and is not presented as inter-rater agreement.

In [1]:
# ruff: noqa: E402
import json
import os
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".jupyter" / "mplconfig"))

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import HTML, Markdown, display

STRICT = ROOT / "data/manual/study_v2_strict_model_audit/strict-model-audit-0fe15fd6b052"
REVIEW = ROOT / "data/manual/study_v2_review_packets/study-v2-review-7cb1b29b5251"
REFERRAL = ROOT / "data/manual/study_v2_referrals/referrals-a4f9bd038101"
CLOCK = ROOT / "data/manual/study_v2_incident_clock/incident-clock-3dc8bb350308"
CONTEXT = ROOT / "data/manual/study_v2_incident_context/incident-context-707a44aafeb4"
CLOSE = ROOT / "data/manual/study_v2_close_cases/close-cases-b175fe03fa80"
DAMAGE = ROOT / "data/manual/study_v2_damage/damage-screening-23c77a57134e"
LAYERS = ROOT / "data/manual/study_v2_layers/study-v2-layers-eed6774fb6c5"
NATIONALITY = ROOT / "data/manual/study_v2_nationality/nationality-diagnostic-2b1b0ffdd961"
GENERATED = ROOT / "reports/generated/study_v2"
GENERATED.mkdir(parents=True, exist_ok=True)

In [2]:
strict_manifest = json.loads((STRICT / "manifest.json").read_text(encoding="utf-8"))
strict_cases = pd.read_csv(STRICT / "strict_model_case_audit.csv", keep_default_na=False)
manifest = json.loads((REVIEW / "manifest.json").read_text(encoding="utf-8"))
review_summary = pd.DataFrame(
    [
        {"packet": "Strict model audit: included decisions", "rows": strict_manifest["included_decisions"]},
        {"packet": "Strict model audit: exclusion checks", "rows": strict_manifest["exclusion_sources"]},
        {"packet": "Strict model audit: corrected included rows", "rows": strict_manifest["corrected_included_rows"]},
        {"packet": "Strict model audit: unavailable public sources", "rows": strict_manifest["model_status_counts"]["model_unresolved_public_evidence"]},
        {"packet": "Reviewer A", "rows": manifest["reviewer_a_rows"]},
        {"packet": "Reviewer B", "rows": manifest["reviewer_b_rows"]},
        {"packet": "Reconciliation queue", "rows": len(pd.read_csv(REVIEW / "reconciliation_queue.csv"))},
    ]
)
display(review_summary)
assert manifest["blind_to_model_final_fields"] is True
assert manifest.get("independent_human_review_complete", False) is False
assert strict_manifest["records_with_fia_citation"] == 920
assert strict_manifest["pending_adversarial"] == 0
assert strict_cases["review_disclosure"].eq("model_led_source_review_not_independent_human_annotation").all()

,packet,rows
0,Strict model audit: included decisions,418
1,Strict model audit: exclusion checks,502
2,Strict model audit: corrected included rows,32
3,Strict model audit: unavailable public sources,4
4,Reviewer A,496
5,Reviewer B,158
6,Reconciliation queue,0


The strict audit covers all 418 included decisions and 502 sampled exclusions. Every record carries
an exact FIA source URL. Thirty-two included rows changed after source review: seven fault labels
and 25 affected-driver lists. Four archive labels remain publicly unavailable and are kept as
unresolved evidence rather than guessed.

Reviewer A and Reviewer B remain blank, blind packets for anyone who later wants independent human
validation. They are not a hidden requirement for reading this model-led portfolio report.